# 13j — single-time-point model diagnostics, **h = 1 only**, two-stage cut

A **horizon-1** variant of `10j_model_diagnostics.ipynb`. Identical in content; it differs in one
config field, `horizons = 1:1`.

**Why it exists.** 10j structurally needs Stage-1 and Stage-2 artefacts at **h1–h4**: it forecasts
over `cfg.horizons`, builds an in-sample fit store from the h4 artefacts, and §2c reconstructs μ
from "the single h4 chain". Crucially, running 10j against h1-only artefacts does **not** fail
loudly — `two_stage_forecast` → `fit_or_load_stage2` **fits when an artefact is missing**, so it
would silently launch full NUTS Stage-1 fits for h = 2, 3, 4. This notebook exists so an h1-only
generation of chains can be diagnosed without that trap.

**What is lost relative to 10j**, stated plainly:

- §1/§1b show a **1-week** forecast fan, not 4, and the in-sample fit is drawn once (from the h=1
  chain) rather than twice (h=1 and h=4);
- §2b's "μ across horizons" collapses to a **single** horizon point per cell — the panel survives
  but carries much less information;
- §2c's timeline is reconstructed from the **h1** chain, so it spans the 8 fit weeks ++ **h1**
  (12 weeks) rather than ++ h1–h4. Still a genuine per-week μ trace from one fit, just a shorter
  forecast reach.

§3 (contact matrices), §4 (degree distributions), §5 (susc/inf) and §6 (dispersion, p⁰) anchor at
the origin week t₀ and read the h=1 chain in 10j too, so they are **unchanged**.

`ORIGIN` is settable via `ENV["ORIGIN_13J"]` (mirroring 12j's `ORIGIN_12J`). Figures go to
**`res/13j/`** — the filenames still carry the `10j_` prefix (it is hard-coded in
`10j_viz_utils.jl`), so the separate directory is what stops them overwriting 10j's own output.

In [ ]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")   # single preamble: base + CoMix pipeline + forecasting framework
using Random, Statistics
mkpath("../res")

# Figure filenames hard-code the "10j_" prefix inside 10j_viz_utils.jl (only tag/file_tag/
# res_dir are caller-controlled), so writing to the default "../res" would OVERWRITE 10j's
# own PNGs for this origin. Every saving call below passes res_dir = RES13.
const RES13 = "../res/13j"
mkpath(RES13)

default_plot_setting()

In [ ]:
# Same config as 8j/9j so the cache keys match. `constant_contacts = false` ⇒ contact degree
# estimated PER WEEK, temporally smoothed by a separable spatio-temporal GP (shared
# ρ_diag/ρ_gap/ρ_time, η, σ_c; scalar intercept c + temporal-level GP + matrix-normal field
# η·(Q·La·z·Ltᵀ), sum-to-zero over the 28 pairs each week). The cached chains this notebook
# reloads were fit under exactly this setting.
# `horizons = 1:1` IS the h1-only lever — every horizon-dependent consumer derives from it
# (H = length(cfg.horizons), win.forecast_weeks, make_mu_horizon_fig's `hz`,
# make_mu_timeline_fig's `h_chain`). It does NOT enter `contacts_label`, and WeeklyWindow's
# all_weeks comes from n_fit/smax alone, so the cache token and the Stage-1 contact window
# are byte-for-byte what a full h1-h4 run would use: artefacts stay forward-compatible.
cfg  = FrameworkConfig(constant_contacts = false, horizons = 1:1)
grid = cis_age_grid()

raw  = load_raw_contact_inputs()
FORECAST_ORIGINS = available_forecast_origins(cfg; grid = grid, craw = raw.craw)
wins = [WeeklyWindow(o; n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons)
        for o in FORECAST_ORIGINS]

# The single forecast date this notebook diagnoses (all four models cached for it at h=1).
# Override with ORIGIN_13J=YYYY-MM-DD, as 12j does with ORIGIN_12J.
ORIGIN = Date(get(ENV, "ORIGIN_13J", "2021-05-09"))
@assert ORIGIN in FORECAST_ORIGINS "ORIGIN $(ORIGIN) not in available origins $(first(FORECAST_ORIGINS))…$(last(FORECAST_ORIGINS))"
win = wins[findfirst(==(ORIGIN), FORECAST_ORIGINS)]

println("origin           : ", ORIGIN)
println("fit weeks        : ", win.fit_weeks[1], " … ", win.fit_weeks[end])
println("forecast weeks   : ", win.forecast_weeks[1], " … ", win.forecast_weeks[end])

In [ ]:
# The four combos (Axis1 degree × Axis2 NGM), fixed order + colour palette; read-only chain viz.
# NOTE: this notebook deliberately stays on the ORIGINAL FOUR, not the six variants that 8j/9j now
# fit. Every diagnostic here reconstructs the contact mean μ from a Stage-1 chain, and neither
# inst/6 baseline adds anything: the NULL model (`no-contact|null`) has no Stage-1 chain at all,
# and the no-interaction model (`unweighted-negbin|mean-diagonal`) SHARES the NegBin Stage-1 chain,
# so its μ is identical to `unweighted-negbin|mean` (only the NGM functional differs downstream).
combos = [(dm, nb) for dm in (NegBinAgePair(), HurdleWeibullAgePair())
                    for nb in (MeanNGM(), NeighbourhoodDegreeNGM())]
labels4    = [string(degree_label(dm), "|", ngm_label(nb)) for (dm, nb) in combos]
model_cols = [:steelblue, :darkorange, :seagreen, :purple]

include("8j_viz_utils.jl")    # stage1_chain_path, stage2_pooled_path, load_transmission_draws, …
include("10j_viz_utils.jl")   # reconstruct_mu_draws (smoothed μ per Stage-1 draw)
println("combos = ", labels4)

In [ ]:
# Reload the cached two-stage artefacts for THIS origin only and assemble the pooled forecast fans
# (NO re-fit). `two_stage_forecast` → `fit_or_load_stage2` reloads each Stage-2 pooled file (10 000
# draws); a missing one would trigger a fallback re-fit (run 8j first). Keyed by model label.
wd    = load_window_data(win; grid = grid)
truth = load_forecast_truth(win; grid = grid)
# this origin's contact/degree windows, one per horizon — a SINGLE window here (h=1).
# Horizon h uses contacts observed at t₀+h (contemporaneous with the target week).
apd_o = [prepare_degree_data(
             WeeklyWindow(ORIGIN + Day(7 * h);
                          n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons),
             cfg; grid = grid, setting = :all,
             df_part_raw = raw.df_part, craw_raw = raw.craw)
         for h in cfg.horizons]

fc_store = Dict{String,Array{Float64,3}}()
for (dm, nb) in combos
    lbl = string(degree_label(dm), "|", ngm_label(nb))
    try
        fc_store[lbl] = two_stage_forecast(dm, nb, wd, cfg, win;
                                           grid = grid, setting = :all,
                                           save_dir = "../dt_intermediate", apd_by_h = apd_o)
    catch err
        @warn "skipped combo (missing/pathological artefact)" model=lbl exception=err
    end
end
println("assembled fans for: ", collect(keys(fc_store)))

In [ ]:
# In-sample expected (fitted) infections over the fit window, per model:
# `fit_window_infection_draws` (10j_viz_utils.jl) reloads the cached horizon-h two-stage artefacts
# (NO re-fit) and returns the MEAN of Stage 2's Normal infection likelihood using OBSERVED lags ⇒
# one-step-ahead fitted mean (columns == win.fit_weeks), NOT the self-iterated forecast. Per pooled
# draw the per-week C* is rebuilt from the Stage-1 chain and paired with the Stage-2 infection draw.
# Built from the h=1 artefacts only — this notebook is h1-only, so there is no h4 companion.
_assemble_fit_store(h) = begin
    store = Dict{String,Array{Float64,3}}()   # A × n_fit × draws, per model
    for (dm, nb) in combos
        r = fit_window_infection_draws(dm, nb, apd_o, wd, cfg, ORIGIN; h = h)
        r === nothing || (store[string(degree_label(dm), "|", ngm_label(nb))] = r)
    end
    store
end
fit_store = _assemble_fit_store(1)
println("assembled fit-window fits (h1): ", collect(keys(fit_store)))

## 1. Forecast point + 90% CI — four ways, single origin

One panel: observed weekly infections (8 fit weeks of history ++ the 4 realised target weeks,
all ages summed), overlaid with, for each of the four models:

- **forecast** (solid line, ○ markers, 90% band) over the 4 forecast weeks — the out-of-sample
  pooled iterated forecast from `two_stage_forecast` (`fc_store`, 10 000 draws), and
- **in-sample fit** (dashed line, ◇ markers, faint 90% band) over the 8 fit weeks — each model's
  **h=1** artefacts' *expected* weekly infections, i.e. the **mean of Stage 2's Normal infection
  likelihood** (`model_transmission`), reconstructed per pooled draw as
  `pred_t = build_ngm(Cstar_m[t], susc, inf, F, wd.antibody[:,t]; gamma_sar)·Σ_s w[s]·wd.I_mean[:,t-s]`,
  where `Cstar_m` is the Stage-1 draw `m = post_index[d]`'s per-week C*. The renewal lags are the
  **observed** infections, so this is the model's one-step-ahead fitted mean (not the self-iterated
  forecast). The h-shift moves only the *contact* window, so the h=1 chain's infection window is the
  8 fit weeks shown here.

Same per-model colour is used for a model's fit and forecast; the dashed/◇ style and the fit weeks
sitting **left of the origin line** keep the in-sample points + CIs visually **separate** from the
forecast points + CIs on the right. A gray dashed legend proxy labels the in-sample style.

**Note on the two bands (deliberate, not an inconsistency):** the **in-sample** band is posterior
uncertainty on the *expected* infection mean (renewal mean with observed lags, **no** observation
noise), whereas the **forecast** band is posterior-predictive (mean **+** `sigma_inf` observation
noise). They answer different questions, so don't read them as the same quantity. The dashed medians
tracking the black observed line indicate good in-sample fit. Dashed line marks the origin.


Only the **h=1** in-sample reconstruction is drawn here (10j draws a second figure from the h=4
artefacts; this notebook has none).

In [ ]:
# §1 forecast point + 90% CI — four ways, single origin (make_forecast_ci_fig, 10j_viz_utils.jl).
# Solid + ○ = self-iterated forecast; dashed + ◇ = in-sample fitted mean, from the h=1 chain.
display(make_forecast_ci_fig(fc_store, fit_store, win, wd, truth, cfg, labels4, model_cols, ORIGIN;
                             fit_h = 1, res_dir = RES13))

## 1b. Forecast point + 90% CI **by age group** — four ways, single origin

The same forecast as §1, but instead of one panel over the age-**summed** total, this breaks the
forecast out **per age group**: one panel per CIS age bin (**2-10 … 70+**), each panel the
single-age analogue of §1. So you can see how the four models track each age group's weekly
infections individually, rather than only in aggregate — the elderly (70+) and youngest bins in
particular have very different levels and are the least-constrained under the (unsmoothed,
independent-per-bin) susc/inf prior (cf. §5), which the summed §1 view hides.

Each panel carries, exactly as in §1:

- **observed** weekly infections for that age (8 fit weeks of history ++ the 4 realised target
  weeks) — black line, ○ markers;
- **forecast** (solid line, ○ markers, 90% band) over the 4 forecast weeks — the out-of-sample
  pooled iterated forecast from `two_stage_forecast` (`fc_store`, 10 000 draws), sliced to that age;
- **in-sample fit** (dashed line, ◇ markers, faint 90% band) over the 8 fit weeks — the model's
  *expected* weekly infections (mean of Stage 2's Normal likelihood, observed lags ⇒ one-step-ahead
  fitted mean), sliced to that age.

Panels use **free y-axes** (age magnitudes differ by orders of magnitude), share the §1 per-model
colour palette, and only the first panel carries the legend. Same two-band caveat as §1 (in-sample
= expected-mean uncertainty, no obs noise; forecast = posterior-predictive, mean **+** `sigma_inf`).

**Caveat (deliberate).** These are **age slices of the joint 7-age forecast**, not seven separate
single-age models. Under the two-stage cut the cached forecast draws propagate **all seven** ages
through the coupled renewal/NGM (ages transmit to each other), so a panel shows age *a*'s marginal
of that joint dynamic — not age *a* fit in isolation.

Drawn once, with the in-sample fit reconstructed from each model's **h=1** artefacts. Saved to
`res/13j/10j_forecast_ci_<origin>_fit-h1_byage.png`.

In [ ]:
# §1b — weekly-infection forecast BY AGE GROUP (four ways), one §1-style panel per CIS age bin.
# Reuses the already-loaded fc_store / fit_store / truth / wd (NO re-fit); each panel is the
# single-age slice of §1's figure (make_forecast_ci_by_age_fig, 10j_viz_utils.jl), in-sample
# fit from the h=1 chain.
# NOTE: age panels are slices of the coupled 7-age forecast, not seven separate single-age models.
display(make_forecast_ci_by_age_fig(fc_store, fit_store, win, wd, truth, cfg, labels4, model_cols,
                                    ORIGIN, grid; fit_h = 1, res_dir = RES13))

## 2. Observed vs GP-smoothed contact mean μ by age pair (mean-NGM models)

For the two **mean-NGM** models, the raw GP-smoothed directional contact mean **μ_{i→j}**
(shown *as-is* — the smoothing part, no (1−p⁰) factor) is reconstructed per posterior draw from
the origin-week slice of the h=1 chain, and compared with the matching observed empirical mean:

- **unweighted-negbin | mean** — μ is the per-capita count mean; observed = `apd.emp_mean`.
- **weighted-hweibull | mean** — μ is the Weibull scale = mean of *positive* duration-weighted
  degrees; observed = `whist_mean(apd.pos_weight)` (collapsed histogram).

Each model → a 2×1 figure overlaying every participant age group, coloured per participant:
**upper = young participants (2-34)**, **lower = older (35+)**; x-axis = contactee age group.
Lines are the smoothed μ (median + 90% ribbon); `×` markers are the observed means.

In [ ]:
# Observed degree data at the origin window (last week == origin). Weekly regime keeps [t,i,j].
apd = prepare_degree_data(WeeklyWindow(ORIGIN; n_fit = cfg.n_fit, smax = cfg.smax,
                                       horizons = cfg.horizons),
                          cfg; grid = grid, setting = :all,
                          df_part_raw = raw.df_part, craw_raw = raw.craw)
t_o = length(apd.weeks)                          # origin week t₀ = last of all_weeks (observed overlay)
@assert apd.weeks[t_o] == win.origin

# This diagnostic anchors at the ORIGIN week t₀: observed contacts at t₀ vs estimated μ at t₀.
# The estimate is read from the h=1 chain, whose contact window now ends at t₀+1 (contacts
# observed h wks ahead), so t₀ is the chain's 2nd-to-last week — column t_o − 1.
t_o_est = t_o - 1                                # h=1-chain GP column corresponding to t₀

# Origin-week diagnostic context, bundled once for the read-only figure helpers (10j_viz_utils.jl):
# §2 make_agepair_fig, §3 make_contactmatrix_fig, §4 make_agepair_ccdf_fig all read from it.
oc = (; apd, t_o, t_o_est, origin = ORIGIN, grid, cfg)

In [ ]:
# unweighted-negbin | mean — smoothed μ is the per-capita count mean (observed = emp_mean)
make_agepair_fig(NegBinAgePair(), MeanNGM(), oc; res_dir = RES13)

In [ ]:
# weighted-hweibull | mean — smoothed μ is the positive-weight mean (observed = whist_mean(pos_weight))
make_agepair_fig(HurdleWeibullAgePair(), MeanNGM(), oc; res_dir = RES13)

## 2b. Contact mean μ from participants 11-15 across horizons h1, by contactee (four models)

Fixing the participant age group at **11-15** (bin 2), how does the model's directional contact
mean **μ_{11-15→j}** to each contactee group *j* evolve across forecast horizons **h1…h4**, and how
does it compare with what was actually observed at each target week?

Each horizon *h* is a **separate re-fit** on a contact window ending at **t₀+h**, so μ (the C\*
slice `q.Cstar[end]` the horizon-*h* forecast is frozen at) genuinely changes with *h*. This reads
that slice via `reconstruct_mu_draws(lbl, ORIGIN, h)` at its **default (last) week** (= t₀+h) —
*not* the t₀ column (`t_o_est`) used in §2–§4. Observed comes from `apd_o[h]` (last window week =
t₀+h): `emp_mean` for negbin, `whist_mean(pos_weight)` for hweibull.

Two figures, one per **degree family** (μ is a per-capita **count** mean for negbin, a positive
**duration-weighted** mean for hweibull — different scales, so kept apart, cf. §2). Within each
figure: **7 contactee panels** (free y), x-axis = horizon; the two coloured lines are the **mean**-
and **neighbourhood**-NGM fits (median + 90% ribbon), `×` = observed. All four models thus appear
across the two figures. Saved to `res/13j/10j_agepair_mu_vs_horizon_<degree>_<origin>.png`.

In [ ]:
# 2b. Contact mean μ from participants 11-15 (bin 2) at horizon h1, per contactee age
#     group, for the four models — split into TWO figures by degree family (μ is a per-capita COUNT
#     mean for negbin, a POSITIVE duration-weighted mean for hweibull: different scales, kept apart).
#     Within each figure the two lines are the mean- vs neighbourhood-NGM fits; × = observed at that
#     forecast week. Horizon h reads μ at the FORECAST week t₀+h (each horizon's own chain default
#     week = the C*-slice its forecast is frozen at); observed from apd_o[h]. See make_mu_horizon_fig.
#     h1-ONLY: `hz` defaults to collect(cfg.horizons) == [1], so each panel carries ONE point.
PART_I = 2                                          # participant age bin "11-15"
@assert grid.LAB[PART_I] == "11-15"
partic_cells = [(PART_I, j, "11-15 → $(grid.LAB[j])") for j in 1:grid.N]
tdesc = "μ from 11-15 at h1 by contactee"

display(make_mu_horizon_fig(NegBinAgePair(), partic_cells, "vs_horizon", tdesc,
                            apd_o, ORIGIN, grid, cfg, labels4, model_cols;
                            res_dir = RES13))                                        # Fig A — count mean
display(make_mu_horizon_fig(HurdleWeibullAgePair(), partic_cells, "vs_horizon", tdesc,
                            apd_o, ORIGIN, grid, cfg, labels4, model_cols;
                            res_dir = RES13))                                        # Fig B — positive weighted mean

### 2b (cont.) — diagonal (self-contact) μ across horizons h1

Companion to the 11-15→contactee view above: the contact-matrix **diagonal** μ_{i→i} — participant
and contactee in the **same** age group *i* (the within-group / self-contact mean) — across forecast
horizons **h1…h4**, for the four models, vs observed.

Same reconstruction as above: each horizon *h* is a separate re-fit, and μ is read at the **forecast
week t₀+h** (default last week of horizon *h*'s own chain) via `reconstruct_mu_draws(lbl, ORIGIN, h)`,
sliced on the diagonal `[:, i, i]`. Observed comes from `apd_o[h]` at the forecast week
(`emp_mean[end,i,i]` for negbin, `whist_mean(pos_weight[end,i,i])` for hweibull). Kept in **two
figures by degree family** with the same colours/legend as above; **7 panels**, one per age group's
self-contact mean. Saved to `res/13j/10j_agepair_mu_diag_vs_horizon_<degree>_<origin>.png`.

In [ ]:
# 2b (cont.). Diagonal (self-contact) μ_{i→i} for each of the 7 age groups at horizon h1
#     — same reconstruction/scales/colours as the 11-15→contactee figure above, but participant AND
#     contactee are the same age group i (the contact-matrix diagonal). Reuses make_mu_horizon_fig
#     with the diagonal cell list.
diag_cells = [(i, i, "$(grid.LAB[i]) → $(grid.LAB[i])") for i in 1:grid.N]
tdesc_diag = "diagonal (self-contact) μ at h1"

display(make_mu_horizon_fig(NegBinAgePair(), diag_cells, "diag_vs_horizon", tdesc_diag,
                            apd_o, ORIGIN, grid, cfg, labels4, model_cols;
                            res_dir = RES13))                                        # Fig A — count mean
display(make_mu_horizon_fig(HurdleWeibullAgePair(), diag_cells, "diag_vs_horizon", tdesc_diag,
                            apd_o, ORIGIN, grid, cfg, labels4, model_cols;
                            res_dir = RES13))                                        # Fig B — positive weighted mean

## 2c. Contact mean μ over the fit window + h1, from the **h1 chain** (11-15 → contactee)

The §2b companion above reads each horizon *h* from its **own** re-fit (chain *h*, at that chain's
forecast week t₀+h). Here instead we take the **single h4 chain** and trace its per-week GP contact
mean **μ_{11-15→j}(t)** across the whole window it spans: the origin's **8 fit weeks** (ending at
t₀) ++ the **4 horizon weeks h1…h4** (ending at t₀+4) — i.e. the h4 window's `all_weeks`. Because
`constant_contacts = false` estimates μ per week, this one fit yields the entire trajectory — the
temporal analogue of §1's h4 in-sample fit.

`reconstruct_mu_draws(lbl, ORIGIN, 4; week_index = t)` reads μ at each window week *t* of the h4
chain; observed comes from that same window `apd_o[4]` per week (`emp_mean` for negbin,
`whist_mean(pos_weight)` for hweibull). x-axis = week (Wed mid-date); the dashed rule marks the
origin t₀ (fit weeks to its left, horizon h1 to its right). Two lines = mean- vs
neighbourhood-NGM (median + 90%); × = observed. Two figures by degree family (count vs
duration-weighted scale, cf. §2b). Saved to `res/13j/10j_agepair_mu_h1timeline_<degree>_<origin>.png`.

In [ ]:
# 2c. μ_{11-15→j}(t) over the fit window + h1, reconstructed from the SINGLE h1 chain (its per-week
#     GP spans the origin's 8 fit weeks ++ the 1 horizon week = 12 weeks with the 4 renewal lags).
#     `h_chain` defaults to maximum(cfg.horizons) == 1, and apd_o[end] == apd_o[1] is the h1 degree
#     window. Shorter forecast reach than 10j's h4 version, but the same per-week μ trace from ONE
#     fit — contrast with §2b, which reads each horizon from its own chain.
tdesc_tl = "μ from 11-15 over fit weeks + h1"

display(make_mu_timeline_fig(NegBinAgePair(), partic_cells, "h1timeline", tdesc_tl,
                             apd_o[end], ORIGIN, grid, cfg, labels4, model_cols;
                             res_dir = RES13))                                          # Fig A — count mean
display(make_mu_timeline_fig(HurdleWeibullAgePair(), partic_cells, "h1timeline", tdesc_tl,
                             apd_o[end], ORIGIN, grid, cfg, labels4, model_cols;
                             res_dir = RES13))                                          # Fig B — positive weighted mean

### 2c (cont.) — diagonal (self-contact) μ_{i→i}(t) from the h1 chain

Same h4-chain trajectory for the contact-matrix **diagonal** μ_{i→i}(t) (participant = contactee),
over the origin's fit weeks ++ horizon h1. Reuses `make_mu_timeline_fig` with the diagonal cell
list; **7 panels**, one per age group's self-contact mean. Saved to
`res/13j/10j_agepair_mu_diag_h1timeline_<degree>_<origin>.png`.

In [ ]:
# 2c (cont.). Diagonal μ_{i→i}(t) from the single h1 chain over the fit window + h1.
tdesc_diag_tl = "diagonal (self-contact) μ over fit weeks + h1"

display(make_mu_timeline_fig(NegBinAgePair(), diag_cells, "diag_h1timeline", tdesc_diag_tl,
                             apd_o[end], ORIGIN, grid, cfg, labels4, model_cols;
                             res_dir = RES13))                                          # Fig A — count mean
display(make_mu_timeline_fig(HurdleWeibullAgePair(), diag_cells, "diag_h1timeline", tdesc_diag_tl,
                             apd_o[end], ORIGIN, grid, cfg, labels4, model_cols;
                             res_dir = RES13))                                          # Fig B — positive weighted mean

## 3. Contact-matrix heatmaps — observed vs estimated (all four combos)

The 7×7 age-pair contact matrix rendered as heatmaps: **observed vs estimated**, one two-panel
`[Observed | Estimated]` figure per combo (**negbin/hweibull × mean/neighbourhood**), the two
panels sharing one colour scale so the fit is directly comparable.

- **Estimated** = median GP-smoothed contact mean **μ_{i→j}** (origin-week slice of the h=1 Stage-1
  chain, reconstructed per posterior draw). Under the two-stage cut, μ is fit **without** infection
  feedback, so it is NGM-independent **and** identical for a degree family's two builders (they share
  one Stage-1 chain); the two mean/neighbourhood figures of a degree family therefore show the same
  μ. We show μ (not the size-biased neighbourhood C\* = ⟨k²⟩/⟨k⟩·g) so observed and estimated stay on
  one comparable scale.
- **Observed** = per-cell empirical mean: negbin → `apd.emp_mean`; hweibull →
  `whist_mean(apd.pos_weight)` (mean of positive duration-weighted degrees; empty cells blank/NaN).

Axes: x = contactee age group j, y = participant age group i (row 1 = youngest, at top).
Saved to `res/13j/10j_contactmatrix_<degree>_<ngm>_<origin>.png`.

In [ ]:
# 7×7 contact-matrix heatmaps: OBSERVED vs ESTIMATED (smoothed μ at the origin week t₀), two panels
# per figure sharing one colour scale — for all four combos (make_contactmatrix_fig, 10j_viz_utils.jl).
for (dm, nb) in combos
    f = make_contactmatrix_fig(dm, nb, oc; res_dir = RES13)
    f === nothing || display(f)
end

## 4. Age-pair degree distribution — observed vs estimated (log-log), mean-NGM models

Beyond the §3 contact-matrix (which compares only the per-cell **mean** μ), this shows the full
**degree distribution** — the project's heavy-tail view — one **7×7 grid** per mean-NGM model:
participant age group *i* (rows) × contactee age group *j* (columns), each a small log-log **CCDF**
panel overlaying the **observed** empirical distribution (● markers) with the **estimated** fitted
distribution (median line + 90% band). The estimated distribution is reconstructed *per posterior
draw* from the **origin-week slice of the h=1 chain** — μ_{i→j} via `reconstruct_mu_draws`,
per-cell dispersion via `reconstruct_dispersion_draws` (block-linear `bl`) — so its **shape**, not
just its mean, is compared with the data.

- **unweighted-negbin | mean** — observed = `apd.dd_count` integer counts; estimated =
  `NegBin(μ_{i→j}, k)` CCDF **conditional on ≥1** (matching the zero-stripped observed CCDF).
- **weighted-hweibull | mean** — observed = positive duration-weighted degrees (`apd.pos_weight`);
  estimated = positive-part `Weibull(κ, λ = μ/Γ(1+1/κ))` CCDF.

Cells with no observed contacts at the origin week render blank. Only the two **mean-NGM** models
are drawn (the NGM builder changes the C\* functional, not the μ/dispersion degree likelihood, so
neighbourhood adds nothing here — cf. §2). Saved to `res/13j/10j_degdist_<degree>_mean_<origin>.png`.

In [ ]:
# unweighted-negbin | mean — observed integer counts vs NegBin CCDF (conditional on ≥1)
make_agepair_ccdf_fig(NegBinAgePair(), MeanNGM(), oc; res_dir = RES13)

In [ ]:
# weighted-hweibull | mean — observed positive duration-weighted degrees vs positive-part Weibull CCDF
make_agepair_ccdf_fig(HurdleWeibullAgePair(), MeanNGM(), oc; res_dir = RES13)

## 5. Relative age-specific susceptibility & infectivity — four ways, single origin

The transmission block estimates an age profile of **relative susceptibility** `susc_a` and
**relative infectivity** `inf_a`, both anchored so the reference age bin **"2-10" (bin 1) = 1** and
the others are multiplicative offsets `exp(softclamp(o_a, log 0.05, log 20))`. The `A-1` non-reference
log-offsets `o_a = sig · z` are **independent per age bin** (`z ~ N(0,1)^{A-1}` iid) — there is
**no cross-bin smoothing** (removed 2026-07-13, user request). The marginal scales
`sig_s, sig_i ~ N⁺(0.5, 0.25²)` are **separately estimated** for susc and inf (loosened 2026-07-13
from `N⁺(0.1, 0.05²)` to admit real age variation).

A shared-length-scale squared-exponential (RBF) GP over the age-bin index (offset `sig · (Lsi · z)`,
`Lsi = chol(K(ρ_si) + 1e-4·I)`, `K_{mn} = exp(-(m-n)²/2ρ_si²)`) previously **correlated** neighbouring
bins; it and its length-scale latent `ρ_si` were dropped. Because that kernel had **unit diagonal**,
removing it leaves each bin's marginal SD unchanged (`= sig`), so with `sig ≈ 0.5` the typical band is
`susc/inf ∈ [0.37, 2.7]` at ±2 SD — and the age profile is **rougher** (adjacent bins are no longer
pulled together). The soft-clamp `[0.05, 20]` (also loosened 2026-07-13 from `[0.2, 5.0]`) remains the
hard safety bound that catches stray Pathfinder draws (the prior's ±2 SD is well interior; clamp at
~±6 SD). (History: GP → RW1 → RW2 → GP → none; see `tasks/lessons.md`.)

Here we read susc/inf straight from each model's **Stage-2 pooled** draws
(`load_transmission_draws` → `pooled.susc`/`pooled.inf`, `N×A` relative to bin 1) and show the
per-age-group **median + 90% band**.

Unlike the GP-smoothed μ (§2–§4), which is **NGM-independent** under the cut, susc/inf are fit in
**Stage 2** conditioning on that model's `C*`, so they genuinely differ across all **four** combos
(negbin/hweibull × mean/neighbourhood). Two panels — **susceptibility | infectivity** — x = age
group, one coloured line per model (same palette as §1). All lines pass through 1 at the reference
bin by construction (gray rule). Read from the **h=1** artefacts. Saved to
`res/13j/10j_susc_inf_<origin>.png`.

In [ ]:
# §5 relative age-specific susceptibility & infectivity — four ways, single origin
# (make_susc_inf_fig, 10j_viz_utils.jl). Reads each model's Stage-2 pooled susc/inf draws
# (N×A, relative to reference bin "2-10" = 1) and plots median + 90% band per age group; two
# panels (susceptibility | infectivity), one coloured line per model. Read from the h=1 artefacts.
display(make_susc_inf_fig(combos, labels4, model_cols, ORIGIN, cfg, grid;
                          h = 1, res_dir = RES13))

## 6. Contact-degree dispersion and fitted hurdle p⁰

Two checks on the **formal-model** additions to Stage 1 (`inst/5_formal_pathfinder_impl.md`, §4.2/§4.3).

**6a — what does the dispersion look like now?** Since 2026-08-02 the dispersion has **no per-cell
random effect**: it is block-linear × week, `log_disp_{ij,t} = m_{bl,t}`, so all ordered cells inside
a child/adult block share one value. The 7×7 map below is therefore **four flat quadrants by
construction** — that is the correct result, not a failure signature. (Between 2026-07-30 and
2026-08-02 a per-cell RE lived here — first a flat hierarchy `τ_t·z_{ij,t}`, then a regularised
horseshoe `τ·λ̃_{ij,t}·z_{ij,t}` — and flat quadrants then meant the RE scale had collapsed. Don't
carry that reading over.) Why it was removed: measured across τ₀ ∈ {0.1, 0.01, 0.005, 0.001}, the
horseshoe's global scale was outbid by the likelihood until τ₀ = 0.001, where the RE vanished
outright — a cliff, not a usable dial. With 49 ordered cells per week and many empty, the per-cell
dispersion was never identified.

The retained `-hd` chains in `dt_intermediate_hierarchical/` record what was given up: the second
panel plots that generation's per-week τ_t, i.e. how large the per-cell scale actually was.
`11j_weekly_identifiability.ipynb` carries the quantitative version (within-block SD, which must be
identically 0 for the current model).

**6b — is the Binomial wired to the right denominator?** The hurdle zero probability `p⁰_{ij,t}` is
**fitted** (weighted path only) with `n_zero ~ Binomial(n, p⁰)` against the participant roster,
rather than plugged in from the empirical `n_zero/n`. This was **retained** through the dispersion
revert. With a flat `Beta(1,1)` and a large roster count the posterior mean must land essentially
**on** the empirical ratio, so the scatter should hug `y = x` for the big cells. Systematic departure
at large `n` means the denominator is wrong. Two legitimate departures: small-`n` cells shrink toward
0.5, and **all-zero cells land just below 1 rather than on it** — which is the point, since it gives
⟨k⟩ > 0 and stops `base_contact`'s `k1 > 0` guard from firing on them.

Both read the **h=1** Stage-1 chain, so they pair with the `apd_o[1]` degree window.

In [ ]:
# §6a — the 7×7 per-cell dispersion at the origin week, with the child/adult block boundary drawn.
# FOUR FLAT QUADRANTS IS THE CORRECT RESULT: dispersion is block-linear × week with no per-cell term
# (inst/3 §4.3), so this is really a 2×2 read-out on a 7×7 grid. One figure per degree family (these
# are Stage-1 latents, so NGM-independent — the two builders of a family share one chain).
for dm in (NegBinAgePair(), HurdleWeibullAgePair())
    lbl = string(degree_label(dm), "|", ngm_label(MeanNGM()))
    p = plot_dispersion_cells(lbl, ORIGIN, cfg, grid; weighted = is_weighted(dm), h = 1,
                              week_index = t_o_est, save_dir = "../dt_intermediate")
    p === nothing || display(p)
end

# What the removed random effect used to be: the previous `-hd` generation DID have a per-week τ_t,
# and its chains are retained in dt_intermediate_hierarchical/. (Measured at 2021-05-09: 1.59–2.54
# negbin, 0.62–1.14 hweibull.) Both the token AND the save_dir differ from the current chains.
for dm in (NegBinAgePair(), HurdleWeibullAgePair())
    lbl = string(degree_label(dm), "|", ngm_label(MeanNGM()))
    p = plot_tau_over_weeks(lbl, ORIGIN, cfg, apd_o[1].weeks; h = 1,
                            contacts = CONTACTS_TOKEN_HD, save_dir = CONTACTS_SAVE_DIR_HD)
    p === nothing || display(p)
end

# §6b — fitted p⁰ vs empirical n₀/n, one point per age-pair cell, size ∝ √(roster count).
# Weighted path ONLY: the NegBin chain has no `p0f` by design (it models its zeros directly), so
# `plot_p0_vs_empirical` returns nothing for it — that is the expected outcome, not a failure.
for dm in (HurdleWeibullAgePair(), NegBinAgePair())
    lbl = string(degree_label(dm), "|", ngm_label(MeanNGM()))
    p = plot_p0_vs_empirical(lbl, ORIGIN, apd_o[1], cfg, grid; h = 1, save_dir = "../dt_intermediate")
    if p === nothing
        println("no fitted p⁰ for $lbl — expected for the NegBin path (zeros modelled directly)")
    else
        display(p)
    end
end

## Notes

- **h = 1 ONLY** (`cfg.horizons = 1:1`) — see the header for exactly what that costs relative
  to 10j. Figures are written to `res/13j/` (filenames keep the hard-coded `10j_` prefix).
- **Single origin**, `ENV["ORIGIN_13J"]`; no re-fit — cached two-stage artefacts reloaded via
  `two_stage_forecast` (pooled fans) and `reconstruct_mu_draws` (smoothed μ from the Stage-1 chain).
- **§2 shows the two mean-NGM models only** (neighbourhood NGM changes the C* functional, not the
  smoothed μ, so it adds nothing to this contact-mean view — and under the cut μ is NGM-independent).
- **μ is shown raw** on each model's own scale (the "smoothing part"): negbin μ = per-capita count
  mean; hweibull μ = mean of positive duration-weighted degrees. No (1−p⁰) per-capita rescaling.
- **§2–§4 anchor at the origin week t₀**: observed contacts at t₀ vs estimated μ at t₀. The estimate
  is the **t₀** slice of the **h=1** Stage-1 chain; because that chain's contact window ends at
  **t₀+1** (contacts observed h wks ahead), t₀ is its **2nd-to-last** week — `week_index = t_o − 1`
  (`t_o_est`), NOT the last. (The last week, t₀+1, is the contact week the h=1 forecast NGM is frozen
  at — see §1's forecast fans.)
- All 7 participant age groups shown, overlaid and coloured; split upper (2-34) / lower (35+);
  x-axis = contactee age group. Lines = smoothed μ (90% ribbon), × = observed.
- **§4 is the full degree distribution** (not just §3's mean): a 7×7 log-log CCDF grid per mean-NGM
  model, observed (●) vs estimated (median + 90% band). The estimated curve is reconstructed per
  Stage-1 draw from the same origin-week (t₀) h=1 slice (μ via `reconstruct_mu_draws`, dispersion
  via `reconstruct_dispersion_draws`). The **negbin** CCDF is normalised **conditional on ≥1** to
  match the zero-stripped observed CCDF (`plot_ccdf!`); the **hweibull** curve is the positive-part
  `Weibull(κ, μ/Γ(1+1/κ))`. Empty cells render blank.